# AgentCore Harness와 MCP(Model Context Protocol) 통합

| 정보 | 세부 내용 |
|---|---|
| 튜토리얼 | Harness - MCP 통합 예제 |
| SDK | boto3 |
| 모델 | Claude Haiku 4.5 (Bedrock) |

**학습 내용:**
- Harness 에이전트를 **MCP 서버**에 연결하는 방법
- 다양한 MCP 공급자(Exa Search, Brave Search 등) 활용
- MCP 서버에 헤더 및 설정 전달
- MCP 연결의 오류 처리 및 디버깅
- MCP 통합 모범 사례

**MCP란 무엇인가요?**

Model Context Protocol(MCP)은 AI 에이전트가 외부 데이터 소스와 도구에 안전하게 연결할 수 있도록 하는 개방형 표준입니다. MCP를 사용하면 에이전트가 다음 작업을 수행할 수 있습니다.
- 웹 검색(Exa, Brave)
- 데이터베이스 접근
- API와 상호 작용
- 사용자 지정 도구 사용

**사전 요구 사항:**
- Amazon Bedrock AgentCore에 접근할 수 있는 AWS 계정
- 자격 증명이 설정된 AWS CLI v2
- Python 3.10+
- 일부 예제에 사용할 MCP 서버 API 키(선택 사항)

## Part 0: 설정

헬퍼 모듈을 가져오고 IAM 실행 역할을 생성합니다.

In [ ]:
import sys
import os
import time
import uuid
from pathlib import Path
import boto3

# 헬퍼
sys.path.insert(0, str(Path.cwd().parent.parent))

# --- 설정 ---
from helper.iam import create_harness_role, delete_harness_role
from helper.client import get_agentcore_control_client, get_agentcore_client

# --- boto3 클라이언트 생성 ---
control = get_agentcore_control_client()
client = get_agentcore_client()

account_id = boto3.client("sts").get_caller_identity()["Account"]
print(f"Account: {account_id}")

IAM 역할 생성

In [ ]:
role_arn = create_harness_role()
print(f"\nExecution Role ARN: {role_arn}")

print("Waiting for IAM role to propagate...")
time.sleep(10)
print("Ready!")

## Part 1: Harness 생성

모든 MCP 예제에서 사용할 AgentCore Harness를 생성합니다.

In [ ]:
HARNESS_NAME = f"MCPIntegration_{uuid.uuid4().hex[:8]}"

resp = control.create_harness(
    harnessName=HARNESS_NAME,
    executionRoleArn=role_arn,
)
harness = resp["harness"]
harness_id = harness["harnessId"]
harness_arn = harness["arn"]
print(f"Harness ID: {harness_id}")
print(f"Harness ARN: {harness_arn}")
print(f"Status: {harness['status']}")

### Harness가 준비될 때까지 대기

In [ ]:
for i in range(12):
    resp = control.get_harness(harnessId=harness_id)
    status = resp["harness"]["status"]
    print(f"Attempt {i + 1}: {status}")
    if status == "READY":
        print("✅ Harness is ready")
        break
    time.sleep(5)

## Part 2: 기본 MCP 통합 - Exa Search

**Exa**는 고품질의 구조화된 웹 검색 결과를 제공하는 AI 기반 검색 엔진입니다.

### Harness에서 MCP를 사용하는 방법

```python
tools=[{
    "type": "remote_mcp",
    "name": "exa",  # 에이전트에 표시되는 도구 이름
    "config": {
        "remoteMcp": {
            "url": "https://mcp.exa.ai/mcp"  # MCP 서버 엔드포인트
        }
    }
}]
```

에이전트는 MCP 서버에서 사용 가능한 도구를 자동으로 검색하고 필요에 따라 사용합니다.

In [ ]:
session_id = str(uuid.uuid4()).upper()
print(f"Session ID: {session_id}\n")

response = client.invoke_harness(
    harnessArn=harness_arn,
    runtimeSessionId=session_id,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "text": "Search for the latest developments in quantum computing in 2024. "
                    "Find 3-5 recent articles and summarize the key breakthroughs."
                }
            ],
        }
    ],
    tools=[
        {
            "type": "remote_mcp",
            "name": "exa",
            "config": {"remoteMcp": {"url": "https://mcp.exa.ai/mcp"}},
        }
    ],
    model={"bedrockModelConfig": {"modelId": "global.anthropic.claude-haiku-4-5-20251001-v1:0"}},
    timeoutSeconds=300,
)

# 응답 스트리밍
for event in response["stream"]:
    if "contentBlockStart" in event:
        start = event["contentBlockStart"].get("start", {})
        if "toolUse" in start:
            tool_name = start["toolUse"].get("name", "?")
            print(f"\n[Tool: {tool_name}]", flush=True)
    elif "contentBlockDelta" in event:
        delta = event["contentBlockDelta"].get("delta", {})
        if "text" in delta:
            print(delta["text"], end="", flush=True)
    elif "messageStop" in event:
        print("\n")
    elif "internalServerException" in event:
        print(f"\nError: {event['internalServerException']}")

## Part 3: 여러 MCP 도구 - 검색 공급자 조합

한 번의 호출에 여러 MCP 도구를 제공할 수 있습니다. 에이전트는 작업에 따라 사용할 도구를 선택합니다.

이 예제에서는 에이전트가 Exa와 다른 MCP 서버에 모두 접근할 수 있게 하고, 어느 것을 사용할지 직접 결정하게 합니다.

In [ ]:
multi_session_id = str(uuid.uuid4()).upper()
print(f"Multi-tool session: {multi_session_id}\n")

response = client.invoke_harness(
    harnessArn=harness_arn,
    runtimeSessionId=multi_session_id,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "text": "Compare the search results from different sources about 'AWS re:Invent 2024 announcements'. "
                    "What were the major announcements?"
                }
            ],
        }
    ],
    tools=[
        {
            "type": "remote_mcp",
            "name": "exa_search",
            "config": {"remoteMcp": {"url": "https://mcp.exa.ai/mcp"}},
        },
        # 참고: 필요에 따라 여기에 다른 MCP 서버 추가
        # {
        #     "type": "remote_mcp",
        #     "name": "brave_search",
        #     "config": {"remoteMcp": {"url": "https://mcp.brave.com/api"}}
        # }
    ],
    model={"bedrockModelConfig": {"modelId": "global.anthropic.claude-haiku-4-5-20251001-v1:0"}},
    timeoutSeconds=300,
)

# 응답 스트리밍
for event in response["stream"]:
    if "contentBlockStart" in event:
        start = event["contentBlockStart"].get("start", {})
        if "toolUse" in start:
            tool_name = start["toolUse"].get("name", "?")
            print(f"\n[Tool: {tool_name}]", flush=True)
    elif "contentBlockDelta" in event:
        delta = event["contentBlockDelta"].get("delta", {})
        if "text" in delta:
            print(delta["text"], end="", flush=True)
    elif "messageStop" in event:
        print("\n")
    elif "internalServerException" in event:
        print(f"\nError: {event['internalServerException']}")

## Part 4: 인증을 사용하는 MCP - 헤더 전달

일부 MCP 서버는 헤더를 통한 인증(API 키, 토큰 등)이 필요합니다.

**참고:** 현재 API에서는 `config` 객체를 통해 헤더를 전달할 수 있습니다. 정확한 형식은 최신 API 문서를 확인하세요.

```python
{
    "type": "remote_mcp",
    "name": "authenticated_mcp",
    "config": {
        "remoteMcp": {
            "url": "https://api.example.com/mcp",
            "headers": {
                "Authorization": "Bearer YOUR_API_KEY",
                "X-Custom-Header": "value"
            }
        }
    }
}
```

**보안 모범 사례:** API 키를 하드 코딩하지 마세요. 환경 변수 또는 AWS Secrets Manager를 사용하세요.

In [ ]:
# 예제: API 키에 환경 변수 사용

# 주석을 해제하고 API 키를 환경 변수로 설정
# export MCP_API_KEY="your-api-key-here"

api_key = os.getenv("MCP_API_KEY", "")

if api_key:
    auth_session_id = str(uuid.uuid4()).upper()
    print(f"Authenticated session: {auth_session_id}\n")

    # 설정 예제(MCP 서버 요구 사항에 맞게 조정)
    tools_config = [
        {
            "type": "remote_mcp",
            "name": "authenticated_search",
            "config": {
                "remoteMcp": {
                    "url": "https://mcp.exa.ai/mcp",
                    # 참고: 헤더 형식은 다를 수 있으므로 최신 API 문서 확인
                    # "headers": {
                    #     "Authorization": f"Bearer {api_key}"
                    # }
                }
            },
        }
    ]

    print("✅ MCP tool configured with authentication")
    print(f"Tool config: {json.dumps(tools_config, indent=2)}")
else:
    print("⚠️  No MCP_API_KEY found in environment. Skipping authenticated example.")
    print("   Set it with: export MCP_API_KEY='your-key-here'")

## Part 5: 오류 처리 및 디버깅

MCP를 사용할 때 다음과 같은 문제가 발생할 수 있습니다.
- 잘못된 MCP 서버 URL
- 네트워크 시간 초과
- 인증 실패
- MCP 서버 오류

오류를 적절하게 처리하는 방법을 살펴보겠습니다.

In [ ]:
def invoke_with_error_handling(harness_arn, session_id, prompt, tools, timeout=300):
    """
    MCP 도구의 오류를 종합적으로 처리하면서 Harness를 호출합니다.
    """
    try:
        response = client.invoke_harness(
            harnessArn=harness_arn,
            runtimeSessionId=session_id,
            messages=[{"role": "user", "content": [{"text": prompt}]}],
            tools=tools,
            model={"bedrockModelConfig": {"modelId": "global.anthropic.claude-haiku-4-5-20251001-v1:0"}},
            timeoutSeconds=timeout,
        )

        result = {"text": "", "tool_uses": [], "errors": []}

        for event in response["stream"]:
            if "contentBlockStart" in event:
                start = event["contentBlockStart"].get("start", {})
                if "toolUse" in start:
                    tool_info = start["toolUse"]
                    result["tool_uses"].append(tool_info.get("name", "unknown"))
                    print(f"\n[Tool: {tool_info.get('name', '?')}]", flush=True)

            elif "contentBlockDelta" in event:
                delta = event["contentBlockDelta"].get("delta", {})
                if "text" in delta:
                    result["text"] += delta["text"]
                    print(delta["text"], end="", flush=True)

            elif "messageStop" in event:
                print("\n")
                stop_reason = event["messageStop"].get("stopReason")
                if stop_reason:
                    print(f"Stop reason: {stop_reason}")

            elif "internalServerException" in event:
                error = event["internalServerException"]
                result["errors"].append(error)
                print(f"\n❌ Error: {error}")

        return result

    except Exception as e:
        print(f"\n❌ Exception: {type(e).__name__}: {str(e)}")
        return {"text": "", "tool_uses": [], "errors": [str(e)]}


# 유효한 MCP로 테스트
debug_session = str(uuid.uuid4()).upper()
print("Testing error handling with valid MCP...\n")

result = invoke_with_error_handling(
    harness_arn=harness_arn,
    session_id=debug_session,
    prompt="Search for 'Amazon Bedrock new features' and summarize the top result.",
    tools=[
        {
            "type": "remote_mcp",
            "name": "exa",
            "config": {"remoteMcp": {"url": "https://mcp.exa.ai/mcp"}},
        }
    ],
    timeout=300,
)

print("\n✅ Invocation complete")
print(f"Tools used: {result['tool_uses']}")
print(f"Errors encountered: {len(result['errors'])}")

### 잘못된 MCP URL로 테스트(오류 처리 예시)

In [ ]:
# 오류 처리 예시이므로 실패할 가능성이 높음
invalid_session = str(uuid.uuid4()).upper()
print("Testing error handling with invalid MCP URL...\n")

result = invoke_with_error_handling(
    harness_arn=harness_arn,
    session_id=invalid_session,
    prompt="Search for 'test query'",
    tools=[
        {
            "type": "remote_mcp",
            "name": "invalid",
            "config": {"remoteMcp": {"url": "https://invalid-mcp-url.example.com/mcp"}},
        }
    ],
    timeout=60,
)

print(f"\nErrors encountered: {result['errors']}")

## Part 6: MCP 통합 모범 사례

### 1. **항상 제한 시간 설정**
MCP 호출은 시간이 걸릴 수 있으며, 특히 웹 검색은 더 오래 걸릴 수 있습니다. 적절한 `timeoutSeconds`를 설정하세요.
```python
timeoutSeconds=300  # 복잡한 검색에는 5분
```

### 2. **안전한 인증 처리**
```python
# ❌ 금지: API 키 하드 코딩
api_key = "sk-abc123..."

# ✅ 권장: 환경 변수 사용
api_key = os.getenv("MCP_API_KEY")

# ✅ 더욱 권장: AWS Secrets Manager 사용
import boto3
secrets = boto3.client('secretsmanager')
api_key = secrets.get_secret_value(SecretId='mcp-api-key')['SecretString']
```

### 3. **명확한 도구 이름 사용**
```python
# ❌ 일반적인 이름
"name": "search"

# ✅ 설명이 포함된 이름
"name": "exa_web_search"
```

### 4. **MCP 사용량 모니터링 및 로깅**
사용 중인 도구와 성공률을 추적하세요.
```python
tool_uses = []
for event in response["stream"]:
    if "contentBlockStart" in event:
        tool_name = event["contentBlockStart"].get("start", {}).get("toolUse", {}).get("name")
        if tool_name:
            tool_uses.append(tool_name)
            print(f"Tool used: {tool_name}")
```

### 5. **MCP 서버를 독립적으로 테스트**
Harness와 통합하기 전에 MCP 서버가 작동하는지 확인하세요.
```bash
curl -X POST https://mcp.exa.ai/mcp \
  -H "Content-Type: application/json" \
  -d '{"method": "tools/list"}'
```

### 6. **MCP를 다른 도구와 조합**
MCP 도구는 기본 제공 Harness 도구와 함께 사용하기 좋습니다.
```python
tools=[
    {"type": "remote_mcp", "name": "exa", "config": {...}},
    {"type": "agentcore_code_interpreter", "name": "code_interpreter"},
    {"type": "agentcore_browser", "name": "browser"},
]
```

## Part 7: 고급 예제 - 연구 어시스턴트

다음 작업을 수행하는 완전한 연구 어시스턴트를 만들어 보겠습니다.
1. MCP를 사용하여 정보 검색
2. 결과 분석
3. 구조화된 보고서 저장

실제 MCP 사용 사례를 보여 주는 예제입니다.

In [ ]:
research_session = str(uuid.uuid4()).upper()
print(f"Research session: {research_session}\n")

research_prompt = """
Research topic: "Generative AI trends in enterprise adoption for 2024"

Please:
1. Search for recent articles and reports about enterprise AI adoption
2. Identify the top 5 trends
3. For each trend, provide:
   - Brief description
   - Key statistics or data points
   - Notable companies or use cases
4. Save the report as a structured JSON file at /tmp/ai_trends_report.json

Format the JSON as:
{
  "topic": "...",
  "date": "...",
  "trends": [
    {
      "name": "...",
      "description": "...",
      "statistics": [...],
      "examples": [...]
    }
  ],
  "sources": [...]
}
"""

response = client.invoke_harness(
    harnessArn=harness_arn,
    runtimeSessionId=research_session,
    messages=[{"role": "user", "content": [{"text": research_prompt}]}],
    tools=[
        {
            "type": "remote_mcp",
            "name": "exa",
            "config": {"remoteMcp": {"url": "https://mcp.exa.ai/mcp"}},
        }
    ],
    model={"bedrockModelConfig": {"modelId": "global.anthropic.claude-haiku-4-5-20251001-v1:0"}},
    timeoutSeconds=300,
)

# 응답 스트리밍
for event in response["stream"]:
    if "contentBlockStart" in event:
        start = event["contentBlockStart"].get("start", {})
        if "toolUse" in start:
            print(f"\n[Tool: {start['toolUse'].get('name', '?')}]", flush=True)
    elif "contentBlockDelta" in event:
        delta = event["contentBlockDelta"].get("delta", {})
        if "text" in delta:
            print(delta["text"], end="", flush=True)
    elif "messageStop" in event:
        print("\n")

### 연구 보고서 가져오기 및 표시

In [ ]:
import json

# 에이전트 VM에서 보고서 가져오기
report_data = ""
resp = client.invoke_agent_runtime_command(
    agentRuntimeArn=harness_arn,
    runtimeSessionId=research_session,
    body={"command": "cat /tmp/ai_trends_report.json 2>/dev/null || echo '{}'"},
)

for event in resp["stream"]:
    if "chunk" in event:
        chunk = event["chunk"]
        if "contentDelta" in chunk and "stdout" in chunk["contentDelta"]:
            report_data += chunk["contentDelta"]["stdout"]

try:
    report = json.loads(report_data)
    print("✅ Research Report Generated:\n")
    print(json.dumps(report, indent=2))
except json.JSONDecodeError:
    print("⚠️  Could not parse report as JSON")
    print(f"Raw output:\n{report_data}")

## 요약

다음 내용을 학습했습니다.
- ✅ Harness 에이전트를 MCP 서버에 연결
- ✅ 한 번의 호출에서 여러 MCP 도구 사용
- ✅ 인증 및 헤더 처리
- ✅ MCP 호출의 오류 처리 구현
- ✅ MCP 통합 모범 사례 적용
- ✅ MCP를 사용한 완전한 연구 어시스턴트 구축

### 다음 단계
1. 다른 MCP 서버(Brave Search, 사용자 지정 MCP 서버 등) 살펴보기
2. 자체 API를 위한 사용자 지정 MCP 서버 구축
3. MCP를 다른 Harness 기능(Memory, Browser, Code Interpreter)과 조합
4. CloudWatch에서 MCP 사용량 및 비용 모니터링

## 리소스 정리

작업을 마치면 요금이 발생하지 않도록 Harness를 삭제합니다.

In [ ]:
control.delete_harness(harnessId=harness_id)
print(f"Deleted harness: {harness_id}")

In [ ]:
# IAM 역할 삭제(다른 예제에서 사용할 수 있으므로, 선택 사항)
delete_harness_role()
print("Deleted IAM role")